In [ ]:
# 05 Experiment C: uncertainty-aware (heteroscedastic Gaussian) regression
#
# Question. Does a mean + log-variance head keep A's point accuracy, do its predicted variances track the actual errors, and do split-conformal intervals built on them reach the target coverage on held-out images?
# Hypothesis H-C (to be tested). Point metrics close to A; Spearman(sigma, |error|) > 0; empirical coverage near 0.90 on images not used for calibration.
# Difference from A. A second linear output (log-variance) and the Gaussian NLL instead of MSE; after training, split-conformal calibration on the 100-image calibration partition. Everything else is identical to A.
# Spec: docs/experiments/c-uncertainty-aware.md. Cell labels: [SAFE] | [TRAINS - OFF by default] | [READS RUN].
# No cell reads the official test set.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ct_iqa.config.loader import load_study_config
from ct_iqa.data.splits import load_manifest
from ct_iqa.experiment import report
from ct_iqa.experiment.grid import apply_overrides
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import read_predictions, run_summary, runs_table, select_run
from ct_iqa.models.weights import sha256_file
import math

from ct_iqa.evaluation.intervals import nonconformity_scores, to_score_scale, uncertainty_error_correlation
from ct_iqa.models.heads import LOGVAR_MAX, LOGVAR_MIN


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Configuration [SAFE]
CONFIG = "configs/experiments/c_uncertainty_aware.yaml"
cfg = load_study_config(CONFIG)        # strict: unknown keys or invalid values raise here, before anything runs
show(f"Configuration: configs/base.yaml + {CONFIG}", report.config_frame(**{cfg.run.experiment_id: cfg.to_dict()}))


In [ ]:
# 2. Outputs, objective and scaling [SAFE]
# The head outputs, in the NORMALIZED 0-1 score space, a mean mu_n = sigmoid(w1 . features) and a log-variance s_n = clamp(w2 . features, -10, 2). The loss is the Gaussian NLL without its constant:
# L_C = 1/(2N) * sum_i [ exp(-s_n,i) * (y_i / 4 - mu_n,i)^2 + s_n,i ]
# Targets are scaled by 1/4 into 0-1 exactly as for A. Back on the raw 0-4 scale: mu = 4 * mu_n and sigma = 4 * exp(s_n / 2). The clamp only guards exp(+/-s) numerically (sigma between 0.03 and 10.9 score units); hitting a bound means the model wanted a more extreme variance. A single deterministic head models aleatoric (data) uncertainty only.
show("Log-variance clamp and its meaning on the 0-4 scale", pd.DataFrame({"s_n": [LOGVAR_MIN, LOGVAR_MAX],
     "sigma (0-4 units) = 4 * exp(s_n / 2)": [4 * math.exp(LOGVAR_MIN / 2), 4 * math.exp(LOGVAR_MAX / 2)]}, index=["lower bound", "upper bound"]).round(4))
print("target coverage (fixed in configs/base.yaml before any result):", cfg.calibration.target_coverage)


In [ ]:
# 3. Preflight [SAFE]
# Checks inputs WITHOUT training and WITHOUT building or running a model: frozen split manifest (hash, disjoint partitions, coverage), pretrained weights SHA-256, image files present (counted, not read), CUDA availability, existing runs. Expected output: a table where every row is True.
labels = json.loads(Path(cfg.data.train_json_path).read_text())
manifest = load_manifest(cfg.data.split_manifest, labels)      # raises if the frozen manifest was altered
sizes = {p: len(manifest[p]) for p in ("train", "validation", "calibration")}
weights = Path(cfg.model.pretrained_weights)
n_images = len(list(Path(cfg.data.train_image_dir).glob("*.tif*")))
PREFLIGHT_PASSED = False      # set True only by the preflight cell below; every training cell checks it
preflight = {
    "split manifest verified (hash, disjoint, complete)": True,
    "partitions are 800 / 100 / 100": list(sizes.values()) == [800, 100, 100],
    "pretrained weights file present": weights.is_file(),
    "pretrained weights SHA-256 matches config": weights.is_file() and sha256_file(weights) == cfg.model.pretrained_sha256,
    "1000 training images present (counted, not read)": n_images == 1000,
}
show("Preflight checks", pd.DataFrame({"passed": preflight}))
show("Environment", pd.DataFrame({"value": {
    "split manifest SHA-256": manifest["manifest_sha256"], "split seed": manifest["seed"],
    "torch": torch.__version__, "CUDA available": torch.cuda.is_available(),
    "CUDA device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}}))
failed_preflight = [k for k, ok in preflight.items() if not ok]
PREFLIGHT_PASSED = not failed_preflight
if failed_preflight:
    raise RuntimeError(f"PREFLIGHT FAILED, training is blocked: {failed_preflight}")
print("PREFLIGHT PASSED: split, weights checksum and image count verified; training cells may be enabled.")


In [ ]:
# 4. Train [TRAINS - OFF by default]
# Expensive: one full training run of the configured epochs per seed, through the shared engine (ct_iqa.experiment.run.run_experiment). Each run gets a NEW run directory and is never overwritten. Progress: one log line per epoch (* = new selected checkpoint) and the run directory when it starts. On an error the run is marked failed in its run_manifest.json and the error is raised here.
# DEVICE is used exactly as written: if it is cuda and CUDA is unavailable the cell stops instead of silently using the CPU. SEEDS defaults to the configured seed; a multi-seed design is a protocol decision that has not been approved yet.
RUN_TRAINING = False
DEVICE = "cuda"
SEEDS = [cfg.train.seed]

if RUN_TRAINING:
    if globals().get("PREFLIGHT_PASSED") is not True:
        raise RuntimeError("Training blocked: the Preflight cell has not run in this kernel or did not pass; run it first, it must finish with PREFLIGHT PASSED")
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(f"DEVICE={DEVICE!r} but CUDA is not available in this kernel; select the GPU kernel or set DEVICE = 'cpu' explicitly")
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    for seed in SEEDS:
        overrides = {"train": {"device": DEVICE, "seed": seed}}
        new_run = run_experiment(apply_overrides(cfg, overrides), config_path=CONFIG, overrides=overrides)
        print(f"COMPLETED: {new_run.as_posix()}  (RUN_ID = {new_run.name!r})")
else:
    print("RUN_TRAINING is False: nothing was trained.")


In [ ]:
# 5. Runs of this experiment [SAFE]
# Every run directory with its status. failed and started (interrupted) runs are listed so that nothing is hidden, but they are never used for results below.
rows = runs_table(cfg.run.experiment_id)
show(f"Runs of {cfg.run.experiment_id}", pd.DataFrame(rows).set_index("run_id").dropna(axis=1, how="all") if rows else pd.DataFrame({"status": {"(none)": "no run yet: PENDING"}}))


In [ ]:
# 6. Select and verify one completed run [READS RUN]
# Set RUN_ID to a run directory name from the table above. With exactly one completed run it may stay None; with several the cell stops rather than guess. Verification: status completed, every artifact matches its recorded SHA-256, history complete, same frozen split and bn_eps as the current config, official test set unused. All later cells are skipped while no run is selected.
RUN_ID = None

run_dir = select_run(cfg.run.experiment_id, RUN_ID)
if run_dir is None:
    print(f"No completed run of {cfg.run.experiment_id}: results are PENDING.")
else:
    S = run_summary(run_dir)                               # raises on failed / incomplete / altered runs
    m, history, metrics = S["manifest"], S["history"], S["metrics"]
    verified = {
        "status is completed": m["status"] == "completed",
        "artifact SHA-256 all match": True,                # run_summary raised otherwise
        "history marked complete": history.get("complete", True),
        "split manifest SHA-256 = current frozen manifest": m["split_manifest_sha256"] == manifest["manifest_sha256"],
        "bn_eps = current config": m["bn_eps"] == cfg.model.bn_eps,
        "official test set unused": m["official_test_set_used"] is False,
    }
    show(f"Verification of {run_dir.as_posix()}", pd.DataFrame({"passed": verified}))
    if not all(verified.values()):
        raise RuntimeError("this run does not match the current protocol; do not interpret it")


In [ ]:
# 7. Selected checkpoint [READS RUN]
# The checkpoint is the epoch with the highest validation PLCC (raw 0-4 scale), evaluated after every epoch; a later epoch replaces it only on a strict improvement. The validation partition therefore SELECTS the reported model: validation metrics below are optimistic (selection-biased).
if run_dir:
    best = next(e for e in history["epochs"] if e["epoch"] == history["best_epoch"])
    show("Checkpoint selection record", pd.DataFrame({"value": {
        "selection policy": history["selection_policy"], "selected epoch (1-based)": history["best_epoch"],
        "of epochs": len(history["epochs"]), "selected validation PLCC": round(history["best_value"], 4),
        "validation PLCC at the final epoch": round(history["epochs"][-1]["val_plcc"], 4),
        "checkpoint file": (run_dir / "checkpoints" / "best.pt").as_posix(),
        "checkpoint SHA-256": m["artifact_sha256"]["checkpoints/best.pt"]}}))


In [ ]:
# 8. Training curves [READS RUN]
# Left: training and validation objective per epoch (normalized 0-1 score space). Right: validation PLCC and SROCC. The dashed line marks the selected epoch.
if run_dir:
    report.plot_history(history, f"{m['run_id']}  (n_train = 800, n_val = 100)")
    plt.show()
    show("Per-epoch history (selected = True: new best checkpoint at that epoch)", report.history_frame(history).round(4))


In [ ]:
# 9. Objective components [READS RUN]
# Each term of the objective separately (train solid, validation dashed) and, for B, the fraction of in-batch pairs that were eligible.
if run_dir:
    report.plot_loss_parts(history, "Objective components per epoch")
    plt.show()
    show("Training-loss components per epoch (sample-weighted batch means)", report.loss_parts_frame(history).round(5))


In [ ]:
# 10. Validation metrics at the selected checkpoint [READS RUN]
# Raw 0-4 score scale, n = 100 validation images. SELECTION-BIASED (the same images chose the epoch); not comparable with historical official-test numbers, which come from a different partition and protocol.
if run_dir:
    show(f"Validation metrics, epoch {metrics['epoch_1based']}, n = {metrics['n']}, {metrics['scale']} (selection-biased)",
         report.metrics_frame(metrics))


In [ ]:
# 11. Prediction and residual diagnostics [READS RUN]
# Scatter against the identity line, residual distribution, residual vs label, and errors per label range with the number of images in each range (ranges with few images are noisy).
if run_dir:
    p = read_predictions(run_dir)
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"])
    report.plot_prediction_scatter(y, f, f"Validation: prediction vs label (n = {len(y)})")
    plt.show()
    report.plot_residuals(y, f, "Validation residuals")
    plt.show()
    show("Error by label range (score units)", report.error_by_score_bin(y, f).round(3))


In [ ]:
# 12. Predicted uncertainty [READS RUN]
# Predicted sigma in raw score units on the 100 validation images, how often the log-variance sits on a clamp bound (validation and, when saved, calibration predictions), and whether larger sigma goes with larger error.
if run_dir:
    _, sigma = to_score_scale(p["prediction_normalized"], p["log_var_normalized"])
    abs_err = np.abs(f - y)
    show("Predicted sigma on validation (score units)", pd.Series(sigma).describe().rename("sigma").to_frame().T.round(4))
    clamp = {"validation": report.clamp_hits(p["log_var_normalized"])}
    cal_csv = run_dir / "predictions_calibration.csv"
    pc = read_predictions(run_dir, cal_csv.name) if cal_csv.is_file() else None
    if pc:
        clamp["calibration"] = report.clamp_hits(pc["log_var_normalized"])
    show("Log-variance clamp-bound hits (evaluation time; training-time hits are not recorded)", pd.DataFrame(clamp).T)
    report.plot_sigma_vs_error(sigma, abs_err, f"Validation: |error| vs predicted sigma (n = {len(sigma)})")
    plt.show()
    print(f"Spearman(sigma, |error|) = {uncertainty_error_correlation(sigma, abs_err):.3f}  (n = {len(sigma)}; > 0 means sigma ranks errors)")
    show("Binned calibration: mean predicted sigma vs observed RMSE (ratio ~ 1 if well calibrated; ~20 images per bin)",
         report.binned_sigma_calibration(sigma, f - y).round(3))


In [ ]:
# 13. Split-conformal intervals [READS RUN]
# Fitted ONLY on the 100-image calibration partition: scores r_i = |y_i - mu_i| / sigma_i, rank k = ceil((n + 1) * 0.9), q = k-th smallest r_i, interval mu +/- q * sigma (also clipped to 0-4). Evaluated on the validation partition, which is disjoint from calibration but was used to select the checkpoint.
# Not a guarantee: the nominal coverage holds only if calibration and evaluation images are exchangeable, which an image-level split without patient identifiers cannot establish; with 100 evaluation images the empirical coverage itself has a standard error of about 0.03. Everything in this section is EXPLORATORY: the final coverage-evaluation partition and rule are an open protocol decision, so no coverage figure from this notebook may be quoted as a result. Which partition carries the final coverage claim is pending the protocol decision.
if run_dir:
    cal = json.loads((run_dir / "calibration.json").read_text())
    q = math.inf if cal["quantile_q"] in ("inf", math.inf) else float(cal["quantile_q"])
    tables = report.calibration_frames(cal)
    show(f"Calibration summary ({cal['fitted_on']} -> {cal['evaluated_on']})", tables["summary"])
    print("EXPLORATORY: calibration was fitted on the calibration partition, but these coverage numbers are measured on the validation partition, which also selected the checkpoint. They are NOT a final coverage result.")
    show("EXPLORATORY coverage and width on validation (selection partition; score units)", tables["intervals"].round(4))
    show("Coverage by predicted-sigma tercile (about 33 images each)", tables["terciles"].round(4))
    if pc:
        mu_c, sig_c = to_score_scale(pc["prediction_normalized"], pc["log_var_normalized"])
        r = nonconformity_scores(pc["label_0_4"], mu_c, sig_c)
        fig, ax = plt.subplots(figsize=(5.5, 3.6))
        ax.hist(r, bins=20, color="tab:blue", alpha=0.8)
        if math.isfinite(q):
            ax.axvline(q, color="k", ls="--", label=f"q = {q:.3f} (k = {cal['rank_k']} of {len(r)})")
        ax.set(xlabel="nonconformity score |y - mu| / sigma", ylabel="calibration images", title="Calibration nonconformity scores")
        ax.legend(); ax.grid(alpha=0.3)
        plt.show()
    else:
        print("predictions_calibration.csv is not in this run (older run): score histogram unavailable.")
    report.plot_intervals(y, f, sigma, q, cal["target_coverage"], tables["terciles"], f"Validation intervals, EXPLORATORY (n = {len(y)})")
    plt.show()


In [ ]:
# 14. Reproducibility record [READS RUN]
# Everything needed to identify and reproduce this run; all paths are relative to the repository root. The full configuration is in config.snapshot.json inside the run directory.
if run_dir:
    show("Reproducibility record", report.reproducibility_frame(S))

# 15. Limitations and next step
# Limitations. Validation point metrics are selection-biased; interval coverage is evaluated on the selection partition; exchangeability is unverified (image-level split, no patient identifiers); sigma is aleatoric only; one seed; the clamp bounds are not tuned.
# Next step. Record the RUN_ID for notebook 06. The final coverage and comparison partition and the seed design are pending the protocol decision (docs/experiment-protocol.md).
